# Partie 4 — Entraînement & optimisation des modèles

**Projet :** prédiction du churn client (Telco Customer Churn) — groupe de 5
**Responsable :** Membre 4
**Livrable attendu :** modèles entraînés + tableau comparatif des expériences

## Objectif de cette partie

Les parties 1 à 3 ont fourni un dataset compris, exploré et préparé. Ici, on **entraîne plusieurs modèles de classification**, on les compare de façon **honnête** (sans fuite de données) et on **optimise** les meilleurs. L'évaluation finale détaillée (matrices de confusion, interprétation) est le travail de la partie 5.

## Ce qu'on retient des parties précédentes

| Constat | Conséquence pour la modélisation |
|---|---|
| 26,5 % de churn seulement (classes déséquilibrées) | l'accuracy est trompeuse : un modèle « toujours No » obtient 73,5 %. On suit surtout **Recall, F1 et ROC-AUC** de la classe churn |
| Contrat, ancienneté, fibre, chèque électronique = facteurs de risque | on pourra vérifier que les modèles retrouvent ces variables |
| `tenure`, `TotalCharges`, `MonthlyCharges` très corrélées | attention aux modèles linéaires (régularisation) |
| Découpage train/test déjà fait (80/20, stratifié, `random_state=42`) | on **réutilise exactement le même découpage** ; le test n'est touché qu'à la toute fin |

## Plan du notebook

1. **Mise en place** : configuration, chargement des données, reconstruction du pipeline de prétraitement
2. Stratégie de validation et modèle de référence
3. Entraînement de plusieurs modèles (Logistic Regression, Decision Tree, Random Forest…)
4. Gestion du déséquilibre des classes
5. Optimisation des hyperparamètres
6. Tableau comparatif des expériences et sauvegarde des modèles

## 1. Mise en place

### 1.1 Imports et configuration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import imblearn
import joblib

# Graine unique pour toute la partie 4 : mêmes résultats à chaque exécution
SEED = 42

# Dossier lu par la partie 4 (produit par la partie 3) et dossiers de sortie
OUT_DIR = Path("outputs")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")

pd.set_option("display.max_columns", 60)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__}")

Python 3.13.7 | pandas 3.1.1 | numpy 2.5.4
scikit-learn 1.9.1 | imbalanced-learn 0.14.2


**Pourquoi afficher les versions ?** Un modèle sauvegardé avec `joblib` ne se recharge correctement qu'avec la même version de scikit-learn. Les versions sont fixées dans `requirements.txt` ; cette cellule permet de vérifier qu'on travaille avec le même environnement que le reste du groupe.